# 05 - Pipeline 全流程诊断

从数据采集到训练验证的每一步可视化，定位模型全黑输出的根源。

1. 数据概览：采集的图像和动作长什么样
2. 相机与 3D 场景：相机在哪、看向哪、射线覆盖什么区域
3. Near/Far 分析：采样点是否覆盖了机器人所在的空间
4. GT 图像 ↔ 3D 射线对应：前景像素对应哪些射线
5. 模型前向逐步追踪：action → EMA state → 3D 场 → 渲染图
6. 梯度与健康检查

In [ ]:
import sys, os, glob
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D
import ipywidgets as widgets
from IPython.display import display, clear_output

%matplotlib inline

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

CUDA_DEVICE = 0
os.environ['CUDA_VISIBLE_DEVICES'] = str(CUDA_DEVICE)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

## 1. 数据概览

In [ ]:
DATA_DIR = os.path.join(PROJECT_ROOT, 'data', 'sequence_data')
files = sorted(glob.glob(os.path.join(DATA_DIR, '*.npz')))
print(f'{len(files)} data files')

# 加载一个文件
data = np.load(files[0])
images = data['images']  # (T, H, W)
actions = data['actions']  # (T, D)
print(f'Images: {images.shape}, range [{images.min():.3f}, {images.max():.3f}]')
print(f'Actions: {actions.shape}, range [{actions.min():.6f}, {actions.max():.6f}]')
print(f'Foreground ratio: {(images > 0.1).mean():.4f} ({(images > 0.1).sum()}/{images.size} pixels)')

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for i, idx in enumerate([0, 125, 250, 375]):
    axes[i].imshow(images[idx], cmap='gray', vmin=0, vmax=1)
    axes[i].set_title(f'Frame {idx}')
    axes[i].axis('off')
plt.suptitle('Sample Frames', fontsize=14)
plt.tight_layout()
plt.show()

# 动作曲线
fig, ax = plt.subplots(figsize=(12, 3))
colors = ['tab:red', 'tab:blue']
for d in range(actions.shape[1]):
    ax.plot(actions[:, d], color=colors[d], label=f'Action {d}')
ax.set_title('Action Curves')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 相机与 3D 场景

可视化相机位置、注视方向、射线覆盖范围、以及机器人大致位置。

In [ ]:
from src.utils.camera import get_rays

# 相机参数
CAM_EYE = np.array([1.5, 0.0, 0.5])
CAM_CENTER = np.array([0.0, 0.0, 0.25])
CAM_UP = np.array([0.0, 0.0, 1.0])
H, W = 100, 100

# 计算焦距 (与数据采集一致: fov=42°)
FOV_DEG = 42.0
focal = 0.5 * W / np.tan(0.5 * FOV_DEG * np.pi / 180)
print(f'Focal length: {focal:.2f} (from FOV={FOV_DEG}°)')

# 生成射线
rays_o, rays_d = get_rays(H, W, focal, CAM_EYE.tolist(), CAM_CENTER.tolist(), CAM_UP.tolist())
rays_o_np = rays_o.numpy()
rays_d_np = rays_d.numpy()

cam_to_center = CAM_CENTER - CAM_EYE
cam_dist = np.linalg.norm(cam_to_center)
print(f'Camera distance to target: {cam_dist:.3f}')
print(f'Camera view direction: {cam_to_center / cam_dist}')
print(f'Rays_o shape: {rays_o_np.shape}, Rays_d shape: {rays_d_np.shape}')
print(f'Ray direction range X: [{rays_d_np[:,0].min():.4f}, {rays_d_np[:,0].max():.4f}]')
print(f'Ray direction range Y: [{rays_d_np[:,1].min():.4f}, {rays_d_np[:,1].max():.4f}]')
print(f'Ray direction range Z: [{rays_d_np[:,2].min():.4f}, {rays_d_np[:,2].max():.4f}]')

In [ ]:
# 3D 场景可视化
fig = plt.figure(figsize=(14, 6))

# --- 左: XY 平面俯视 ---
ax1 = fig.add_subplot(121)

# 画射线 (降采样)
sample_rays = np.random.choice(len(rays_o_np), 200, replace=False)
NEAR, FAR = 0.5, 2.5
for i in sample_rays:
    p_near = rays_o_np[i] + rays_d_np[i] * NEAR
    p_far = rays_o_np[i] + rays_d_np[i] * FAR
    ax1.plot([p_near[0], p_far[0]], [p_near[1], p_far[1]], 'b-', alpha=0.05, linewidth=0.5)

# 相机
ax1.plot(*CAM_EYE[:2], 'ro', markersize=10, label='Camera')
ax1.annotate('Camera', CAM_EYE[:2], textcoords='offset points', xytext=(5, 5))

# 注视点
ax1.plot(*CAM_CENTER[:2], 'g^', markersize=10, label='Look-at')

# 机器人近似位置 (杆体 z=0~0.5, x,y≈0, 半径≈0.015, 弯曲后偏移约±0.05)
circle = plt.Circle((0, 0), 0.05, fill=False, color='orange', linewidth=2, label='Robot (approx)')
ax1.add_patch(circle)

# Near/Far 面上的采样点
for t_val, label, color in [(NEAR, 'Near', 'red'), (FAR, 'Far', 'blue')]:
    corners = []
    for ri in [0, H-1]:
        for ci in [0, W-1]:
            idx = ri * W + ci
            pt = rays_o_np[idx] + rays_d_np[idx] * t_val
            corners.append(pt[:2])
    corners = np.array(corners)
    from scipy.spatial import ConvexHull
    hull = ConvexHull(corners)
    for simplex in hull.simplices:
        ax1.plot(corners[simplex, 0], corners[simplex, 1], f'{color[0]}-', alpha=0.8, linewidth=2)

ax1.set_xlim(-1, 2.5)
ax1.set_ylim(-1.5, 1.5)
ax1.set_xlabel('X')
ax1.set_ylabel('Y')
ax1.set_title('Top View (XY plane)')
ax1.set_aspect('equal')
ax1.legend()
ax1.grid(True, alpha=0.3)

# --- 右: XZ 侧面 ---
ax2 = fig.add_subplot(122)
for i in sample_rays:
    p_near = rays_o_np[i] + rays_d_np[i] * NEAR
    p_far = rays_o_np[i] + rays_d_np[i] * FAR
    ax2.plot([p_near[0], p_far[0]], [p_near[2], p_far[2]], 'b-', alpha=0.05, linewidth=0.5)

ax2.plot(CAM_EYE[0], CAM_EYE[2], 'ro', markersize=10, label='Camera')
ax2.plot(CAM_CENTER[0], CAM_CENTER[2], 'g^', markersize=10, label='Look-at')

# 机器人 (z=0~0.5)
ax2.plot([0, 0], [0, 0.5], 'orange', linewidth=4, label='Robot (straight)')

for t_val, label, color in [(NEAR, 'Near', 'red'), (FAR, 'Far', 'blue')]:
    corners = []
    for ri in [0, H-1]:
        for ci in [0, W-1]:
            idx = ri * W + ci
            pt = rays_o_np[idx] + rays_d_np[idx] * t_val
            corners.append(pt[[0, 2]])
    corners = np.array(corners)
    hull = ConvexHull(corners)
    for simplex in hull.simplices:
        ax2.plot(corners[simplex, 0], corners[simplex, 1], f'{color[0]}-', alpha=0.8, linewidth=2)

ax2.set_xlim(-1, 2.5)
ax2.set_ylim(-0.5, 1.5)
ax2.set_xlabel('X')
ax2.set_ylabel('Z')
ax2.set_title('Side View (XZ plane)')
ax2.set_aspect('equal')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Camera & Scene Geometry', fontsize=14)
plt.tight_layout()
plt.show()

print(f'Near plane (red): distance {NEAR} from camera')
print(f'Far plane (blue): distance {FAR} from camera')
print(f'Robot is at distance ~{cam_dist:.2f} from camera (between near={NEAR} and far={FAR})')

## 3. Near/Far 分析

关键问题：射线的采样点是否落在机器人所在的 3D 区域？

机器人（Cosserat 杆）:
- 底部固定在原点 (0, 0, 0)
- 自然状态向 Z 正方向延伸到 z ≈ 0.5
- 弯曲时 x,y 偏移约 ±0.05
- 半径 0.015m（非常细）

In [ ]:
# 中心射线（穿过图像中心的射线）沿途的采样点
center_ray_idx = (H // 2) * W + (W // 2)
center_o = rays_o_np[center_ray_idx]
center_d = rays_d_np[center_ray_idx]

n_samples = 64
t_vals = np.linspace(0, 1, n_samples)
z_vals = NEAR * (1 - t_vals) + FAR * t_vals
points = center_o + center_d * z_vals[:, None]

print(f'Center ray origin: {center_o}')
print(f'Center ray direction: {center_d}')
print(f'')
print(f'Sampling points along center ray:')
print(f'  Near (t=0): {points[0]}')
print(f'  Mid  (t=0.5): {points[n_samples//2]}')
print(f'  Far  (t=1): {points[-1]}')

# 与机器人中心 (0,0,0.25) 的距离
robot_center = np.array([0, 0, 0.25])
dists_to_robot = np.linalg.norm(points - robot_center, axis=1)
closest_idx = np.argmin(dists_to_robot)
print(f'')
print(f'Closest point to robot center (0,0,0.25):')
print(f'  Point: {points[closest_idx]}, distance: {dists_to_robot[closest_idx]:.4f}')
print(f'  At z_val = {z_vals[closest_idx]:.4f}')

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

# 左: XZ 侧面，采样点分布
ax2_data = []
for i in range(0, len(rays_o_np), 500):  # 每隔 500 条射线
    pts = rays_o_np[i] + rays_d_np[i] * z_vals[:, None]
    ax2_data.append(pts)
ax2_data = np.concatenate(ax2_data)
ax1.scatter(ax2_data[:, 2], ax2_data[:, 0], s=0.1, alpha=0.1, c='blue')
ax1.axhline(y=0, color='orange', linewidth=2, label='Robot base (x=0)')
ax1.axhline(y=0.5, color='orange', linewidth=1, linestyle='--', label='Robot tip (x=0.5)')
ax1.scatter(robot_center[0], robot_center[2], c='red', s=100, marker='*', label='Robot center', zorder=5)
ax1.plot(center_o[2], center_o[0], 'ko', markersize=8, label='Camera')
ax1.set_xlabel('Z')
ax1.set_ylabel('X')
ax1.set_title('Sampling Point Distribution (Side View)')
ax1.legend()
ax1.grid(True, alpha=0.3)

# 右: 中心射线沿途到机器人的距离
ax2.plot(z_vals, dists_to_robot, 'b-', linewidth=2)
ax2.axvline(x=z_vals[closest_idx], color='r', linestyle='--', label=f'Closest: d={dists_to_robot[closest_idx]:.4f}')
ax2.axhline(y=0.015, color='orange', linestyle=':', label='Rod radius (0.015)')
ax2.set_xlabel('Depth (z_val)')
ax2.set_ylabel('Distance to robot center (0,0,0.25)')
ax2.set_title('Center Ray: Distance to Robot vs Depth')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Near/Far 参数扫描：看看不同 near/far 对应射线穿过哪些区域
fig, axes = plt.subplots(1, 4, figsize=(20, 5))
near_far_pairs = [(0.5, 2.5), (0.3, 2.0), (1.0, 2.0), (1.0, 1.5)]

for ax, (nr, fr) in zip(axes, near_far_pairs):
    t_vals = np.linspace(0, 1, 64)
    zv = nr * (1 - t_vals) + fr * t_vals
    pts = center_o + center_d * zv[:, None]
    ax.scatter(pts[:, 2], pts[:, 0], c=zv, cmap='viridis', s=10)
    ax.scatter(robot_center[0], robot_center[2], c='red', s=100, marker='*', zorder=5)
    ax.set_xlabel('Z')
    ax.set_ylabel('X')
    ax.set_title(f'near={nr}, far={fr}')
    ax.grid(True, alpha=0.3)

plt.suptitle('Near/Far Sweep: Center Ray Sampling Points', fontsize=14)
plt.tight_layout()
plt.show()

## 4. GT 图像 ↔ 3D 射线对应

哪些射线穿过了机器人（前景像素），它们在 3D 中的路径是什么？

In [ ]:
# 选择一帧，看前景像素对应的射线
frame_idx = 250
gt_img = images[frame_idx]  # (H, W)
gt_flat = gt_img.flatten()  # (H*W,)

fg_mask = gt_flat > 0.1
fg_indices = np.where(fg_mask)[0]
bg_indices = np.where(~fg_mask)[0]

print(f'Frame {frame_idx}: {len(fg_indices)} foreground pixels ({len(fg_indices)/len(gt_flat)*100:.1f}%)')

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# GT 图像
axes[0].imshow(gt_img, cmap='gray', vmin=0, vmax=1)
axes[0].set_title(f'GT Frame {frame_idx}')
axes[0].axis('off')

# 前景掩码
mask_2d = fg_mask.reshape(H, W)
axes[1].imshow(mask_2d, cmap='gray')
axes[1].set_title(f'Foreground Mask ({len(fg_indices)} px)')
axes[1].axis('off')

# 前景射线在图像中的位置 (像素坐标)
fg_rows = fg_indices // W
fg_cols = fg_indices % W
axes[2].scatter(fg_cols, fg_rows, s=2, c='white')
axes[2].set_facecolor('black')
axes[2].set_xlim(0, W)
axes[2].set_ylim(H, 0)
axes[2].set_title('Foreground Ray Positions')
axes[2].set_aspect('equal')

plt.tight_layout()
plt.show()

In [ ]:
# 前景射线 vs 背景射线的 3D 路径
fig = plt.figure(figsize=(16, 6))

ax1 = fig.add_subplot(121)
ax2 = fig.add_subplot(122)

n_show = min(50, len(fg_indices))
fg_show = np.random.choice(fg_indices, n_show, replace=False)
bg_show = np.random.choice(bg_indices, n_show, replace=False)

t_vals = np.linspace(0, 1, 64)
z_vals = NEAR * (1 - t_vals) + FAR * t_vals

for idx in fg_show:
    pts = rays_o_np[idx] + rays_d_np[idx] * z_vals[:, None]
    ax1.plot(pts[:, 2], pts[:, 0], 'r-', alpha=0.2, linewidth=0.5)
for idx in bg_show:
    pts = rays_o_np[idx] + rays_d_np[idx] * z_vals[:, None]
    ax1.plot(pts[:, 2], pts[:, 0], 'b-', alpha=0.1, linewidth=0.5)

ax1.scatter(robot_center[2], robot_center[0], c='orange', s=100, marker='*', zorder=5)
ax1.set_xlabel('Z')
ax1.set_ylabel('X')
ax1.set_title('Red=Foreground rays, Blue=Background rays')
ax1.grid(True, alpha=0.3)

# 近距离看前景射线穿过的区域
for idx in fg_show:
    pts = rays_o_np[idx] + rays_d_np[idx] * z_vals[:, None]
    ax2.plot(pts[:, 1], pts[:, 0], 'r-', alpha=0.3, linewidth=0.5)

ax2.set_xlabel('Y')
ax2.set_ylabel('X')
ax2.set_title('Foreground Rays (XY plane, close-up)')
ax2.set_xlim(-0.15, 0.15)
ax2.set_ylim(-0.15, 0.15)
ax2.set_aspect('equal')
ax2.grid(True, alpha=0.3)

plt.suptitle('Foreground vs Background Rays in 3D', fontsize=14)
plt.tight_layout()
plt.show()

## 5. 模型前向逐步追踪

加载模型，对一帧数据跑一次完整前向，观察每一步的中间输出。

In [ ]:
from src.models.model_mstnf import MSTNFModel
from src.data.dataset import SoftSequenceDataset
from src.utils.rendering import OM_rendering, sample_stratified

# 加载最新实验
LOG_BASE = os.path.join(PROJECT_ROOT, 'train_log', 'train_mstnf')
exp_dirs = sorted([d for d in os.listdir(LOG_BASE) if d.startswith('exp_')])
if exp_dirs:
    EXP_DIR = os.path.join(LOG_BASE, exp_dirs[-1])
    print(f'Using experiment: {exp_dirs[-1]}')
else:
    # fallback
    EXP_DIR = LOG_BASE
    print(f'No exp dirs found, using base: {LOG_BASE}')

# 加载配置
import json
config_path = os.path.join(EXP_DIR, 'config.json')
if os.path.exists(config_path):
    with open(config_path) as f:
        saved_config = json.load(f)
    print(f'Config: window={saved_config.get("window_size")}, '
          f'scales={saved_config.get("n_scales")}, '
          f'hidden={saved_config.get("hidden_dim")}')
else:
    saved_config = {'window_size': 20, 'n_scales': 4, 'hidden_dim': 128,
                    'density_bias': -1.0}
    print('No config.json, using defaults')

# 加载 norm factor
norm_path = os.path.join(EXP_DIR, 'action_norm_factor.txt')
norm_factor = float(np.loadtxt(norm_path)) if os.path.exists(norm_path) else 1.0
print(f'Norm factor: {norm_factor}')

In [ ]:
# 加载数据和模型
val_file = files[-1]
SEQ_LEN = saved_config.get('window_size', 20)
val_ds = SoftSequenceDataset(DATA_DIR, seq_len=SEQ_LEN, file_list=[val_file], norm_factor=norm_factor)
print(f'Val data: {len(val_ds)} frames, action_dim={val_ds.action_dim}')

action_dim = val_ds.action_dim
model = MSTNFModel(
    action_dim=action_dim,
    window_size=SEQ_LEN,
    n_scales=saved_config.get('n_scales', 4),
    hidden_dim=saved_config.get('hidden_dim', 128),
).to(device)

# 加载权重
weight_path = os.path.join(EXP_DIR, 'model', 'best_model.pt')
if not os.path.exists(weight_path):
    # 尝试其他权重文件
    import glob as g
    weight_files = sorted(g.glob(os.path.join(EXP_DIR, 'model', '*.pt')))
    weight_path = weight_files[-1] if weight_files else None
    print(f'best_model.pt not found, using {os.path.basename(weight_path)}')

if weight_path and os.path.exists(weight_path):
    model.load_state_dict(torch.load(weight_path, map_location=device))
    print(f'Loaded: {weight_path}')
else:
    print('WARNING: No model weights found, using random init')

model.eval()
decays = model.get_learned_decays()
print(f'Learned decays: {[f"{d:.4f}" for d in decays]}')
print(f'Density bias: {model.decoder.net[-1].bias.data[1].item():.4f}')

In [ ]:
# Step 5a: EMA 编码分析
# 取一帧数据，看 EMA 编码的输出
frame_idx = 250
action_window, gt_frame = val_ds[frame_idx]
action_window = action_window.unsqueeze(0).to(device)  # (1, K, D)
gt_frame_np = gt_frame.reshape(H, W).numpy()

print(f'Action window shape: {action_window.shape}')
print(f'Action range in window: [{action_window.min().item():.4f}, {action_window.max().item():.4f}]')
print(f'Norm factor: {norm_factor} → raw actions ≈ [{action_window.min().item()*norm_factor:.6f}, {action_window.max().item()*norm_factor:.6f}]')

with torch.no_grad():
    # Step 1: EMA 编码
    physics_state = model.encode_temporal(action_window)  # (1, Hidden)
    current_action = action_window[:, -1, :]  # (1, D)

print(f'')
print(f'Physics state shape: {physics_state.shape}')
print(f'Physics state stats: mean={physics_state.mean().item():.6f}, '
      f'std={physics_state.std().item():.6f}, '
      f'range=[{physics_state.min().item():.6f}, {physics_state.max().item():.6f}]')
print(f'Current action: {current_action.cpu().numpy()})')

# 检查 EMA 各尺度的权重分布
decays_tensor = model.temporal.decays
K = action_window.shape[1]
powers = torch.arange(K, device=device).float()
weights = decays_tensor.unsqueeze(1) ** (K - 1 - powers).unsqueeze(0)
weights_norm = weights / weights.sum(dim=1, keepdim=True)

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
for s in range(len(decays_tensor)):
    axes[0].plot(weights_norm[s].cpu().numpy(), label=f'Scale {s} (decay={decays[s]:.3f})')
axes[0].set_xlabel('Time step (0=oldest)')
axes[0].set_ylabel('Weight')
axes[0].set_title('EMA Weights per Scale')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# 动作窗口内容
raw_actions = action_window[0].cpu().numpy() * norm_factor
for d in range(action_dim):
    axes[1].plot(raw_actions[:, d], label=f'Action {d}')
axes[1].set_xlabel('Time step')
axes[1].set_ylabel('Action value (denormalized)')
axes[1].set_title('Action Window')
axes[1].legend()
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Step 5b: 3D 空间场查询
# 在机器人附近构建密集网格，查询 density 和 visibility

# 粗网格覆盖机器人工作空间
GRID_RES = 30
xs = np.linspace(-0.1, 0.1, GRID_RES)
ys = np.linspace(-0.1, 0.1, GRID_RES)
zs = np.linspace(0.0, 0.55, GRID_RES)
gx, gy, gz = np.meshgrid(xs, ys, zs, indexing='ij')
grid_pts = np.stack([gx.flatten(), gy.flatten(), gz.flatten()], axis=-1)
grid_tensor = torch.tensor(grid_pts, dtype=torch.float32, device=device)

print(f'Query grid: {GRID_RES}^3 = {len(grid_pts)} points')
print(f'Grid range: x=[{xs[0]:.2f},{xs[-1]:.2f}], y=[{ys[0]:.2f},{ys[-1]:.2f}], z=[{zs[0]:.2f},{zs[-1]:.2f}]')

with torch.no_grad():
    state_exp = physics_state.expand(len(grid_tensor), -1)
    action_exp = current_action.expand(len(grid_tensor), -1)
    
    chunk = 4096
    raw_parts = []
    for i in range(0, len(grid_tensor), chunk):
        pts = grid_tensor[i:i+chunk].unsqueeze(1)  # (chunk, 1, 3)
        s = state_exp[i:i+chunk]
        a = action_exp[i:i+chunk]
        raw_parts.append(model.decode_spatial(pts, s, a))
    raw = torch.cat(raw_parts, dim=0).squeeze(1)  # (N, 2)

visibility = raw[:, 0].cpu().numpy()
density = raw[:, 1].cpu().numpy()
alpha = 1.0 - np.exp(-np.maximum(density, 0))

print(f'')
print(f'Raw output stats:')
print(f'  Visibility: [{visibility.min():.4f}, {visibility.max():.4f}], mean={visibility.mean():.4f}')
print(f'  Density:    [{density.min():.4f}, {density.max():.4f}], mean={density.mean():.4f}')
print(f'  Alpha:      [{alpha.min():.4f}, {alpha.max():.4f}], mean={alpha.mean():.4f}')
print(f'  Points with alpha > 0.01: {(alpha > 0.01).sum()} / {len(alpha)}')
print(f'  Points with density > 0:  {(density > 0).sum()} / {len(density)}')

In [ ]:
# 3D 场可视化
fig = plt.figure(figsize=(18, 12))

# 左上: density 分布 (XZ 切片, y=0)
ax1 = fig.add_subplot(221)
mid_y = GRID_RES // 2
density_xz = density.reshape(GRID_RES, GRID_RES, GRID_RES)[:, mid_y, :]
im1 = ax1.imshow(density_xz.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                 origin='lower', cmap='RdBu_r', aspect='auto')
ax1.set_xlabel('X')
ax1.set_ylabel('Z')
ax1.set_title('Density (XZ slice, y=0)')
plt.colorbar(im1, ax=ax1)

# 右上: alpha 分布
ax2 = fig.add_subplot(222)
alpha_xz = alpha.reshape(GRID_RES, GRID_RES, GRID_RES)[:, mid_y, :]
im2 = ax2.imshow(alpha_xz.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                 origin='lower', cmap='hot', aspect='auto', vmin=0, vmax=1)
ax2.set_xlabel('X')
ax2.set_ylabel('Z')
ax2.set_title('Alpha = 1-exp(-relu(density)) (XZ slice, y=0)')
plt.colorbar(im2, ax=ax2)

# 左下: visibility 分布
ax3 = fig.add_subplot(223)
vis_xz = visibility.reshape(GRID_RES, GRID_RES, GRID_RES)[:, mid_y, :]
im3 = ax3.imshow(vis_xz.T, extent=[xs[0], xs[-1], zs[0], zs[-1]],
                 origin='lower', cmap='gray', aspect='auto')
ax3.set_xlabel('X')
ax3.set_ylabel('Z')
ax3.set_title('Visibility (XZ slice, y=0)')
plt.colorbar(im3, ax=ax3)

# 右下: 直方图
ax4 = fig.add_subplot(224)
ax4.hist(density, bins=100, alpha=0.7, label='Density')
ax4.hist(visibility, bins=100, alpha=0.7, label='Visibility')
ax4.axvline(x=0, color='r', linestyle='--', label='density=0 threshold')
ax4.set_title('Raw Output Distribution')
ax4.legend()
ax4.set_yscale('log')

plt.suptitle(f'3D Field Analysis (Frame {frame_idx})', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Step 5c: 完整渲染 vs GT
rays_o_dev = rays_o.to(device)
rays_d_dev = rays_d.to(device)

with torch.no_grad():
    # 全图渲染
    B = 1
    n_samp = 64
    pts, z_vals = sample_stratified(rays_o_dev, rays_d_dev, NEAR, FAR, n_samp, perturb=False)
    
    N_rays = pts.shape[0]
    state_exp = physics_state.expand(N_rays, -1)
    action_exp = current_action.expand(N_rays, -1)
    pts_exp = pts.unsqueeze(0).expand(B, -1, -1, -1).reshape(-1, n_samp, 3)
    state_exp_full = state_exp.unsqueeze(0).expand(B, N_rays, -1).reshape(-1, physics_state.shape[-1])
    action_exp_full = action_exp.unsqueeze(0).expand(B, N_rays, -1).reshape(-1, action_dim)
    
    chunk = 4096
    raw_parts = []
    for i in range(0, pts_exp.shape[0], chunk):
        raw_parts.append(model.decode_spatial(
            pts_exp[i:i+chunk], state_exp_full[i:i+chunk], action_exp_full[i:i+chunk]))
    raw = torch.cat(raw_parts, dim=0).reshape(B, N_rays, n_samp, 2)
    
    rgb_map, alpha_map = OM_rendering(raw.reshape(-1, n_samp, 2))
    pred_img = rgb_map.reshape(H, W).cpu().numpy()
    alpha_img = alpha_map.sum(dim=1).reshape(H, W).cpu().numpy()

fig, axes = plt.subplots(1, 4, figsize=(20, 5))

axes[0].imshow(gt_frame_np, cmap='gray', vmin=0, vmax=1)
axes[0].set_title('GT')
axes[0].axis('off')

axes[1].imshow(pred_img, cmap='gray', vmin=0, vmax=1)
axes[1].set_title(f'Prediction (range [{pred_img.min():.4f}, {pred_img.max():.4f}])')
axes[1].axis('off')

axes[2].imshow(alpha_img, cmap='hot', vmin=0)
axes[2].set_title(f'Cumulative Alpha (range [{alpha_img.min():.4f}, {alpha_img.max():.4f}])')
axes[2].axis('off')

diff = np.abs(pred_img - gt_frame_np)
axes[3].imshow(diff, cmap='hot', vmin=0, vmax=1)
axes[3].set_title(f'|Pred - GT| (MSE={np.mean(diff**2):.6f})')
axes[3].axis('off')

plt.suptitle(f'Full Render vs GT (Frame {frame_idx})', fontsize=14)
plt.tight_layout()
plt.show()

## 6. 梯度与健康检查

检查模型各层的梯度大小，判断是否有梯度消失。

In [ ]:
# 梯度检查：用一个 batch 做一次完整前向 + 反向
from torch.utils.data import DataLoader

# 重新加载为 pairs 模式
train_ds_pairs = SoftSequenceDataset(DATA_DIR, seq_len=SEQ_LEN, file_list=[files[0]],
                                     norm_factor=norm_factor, return_pairs=True)
train_loader = DataLoader(train_ds_pairs, batch_size=4, shuffle=True)

model.train()
seq_t, seq_t1, img_t, img_t1 = next(iter(train_loader))
seq_t = seq_t.to(device)
img_t = img_t.to(device)

# 前向 (与训练一致，前景过采样)
B, K, D = seq_t.shape
physics_state = model.encode_temporal(seq_t)
current_action = seq_t[:, -1, :]

# 前景过采样
fg_mask = img_t[0] > 0.1
fg_idx = torch.where(fg_mask)[0]
n_rays_sample = 1024
n_fg = int(n_rays_sample * 0.5)
n_bg = n_rays_sample - n_fg
if len(fg_idx) > 0:
    chosen_fg = fg_idx[torch.randint(len(fg_idx), (n_fg,), device=device)]
    chosen_bg = torch.randint(len(rays_o_dev), (n_bg,), device=device)
    sel = torch.cat([chosen_fg, chosen_bg])
else:
    sel = torch.randint(len(rays_o_dev), (n_rays_sample,), device=device)

pts, z_vals = sample_stratified(rays_o_dev[sel], rays_d_dev[sel], NEAR, FAR, 64)
N_rays = pts.shape[0]
state_exp = physics_state.unsqueeze(1).expand(-1, N_rays, -1).reshape(-1, physics_state.shape[-1])
action_exp = current_action.unsqueeze(1).expand(-1, N_rays, -1).reshape(-1, D)
pts_exp = pts.unsqueeze(0).expand(B, -1, -1, -1).reshape(-1, 64, 3)

chunk = 4096
raw_parts = []
for i in range(0, pts_exp.shape[0], chunk):
    raw_parts.append(model.decode_spatial(pts_exp[i:i+chunk], state_exp[i:i+chunk], action_exp[i:i+chunk]))
raw = torch.cat(raw_parts, dim=0).reshape(B, N_rays, 64, 2)
rgb_map, _ = OM_rendering(raw.reshape(-1, 64, 2))
pred = rgb_map.reshape(B, -1)

gt_sampled = img_t[:, sel]
loss = torch.nn.functional.mse_loss(pred, gt_sampled)

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
optimizer.zero_grad()
loss.backward()

print(f'Loss: {loss.item():.6f}')
print(f'Pred range: [{pred.min().item():.6f}, {pred.max().item():.6f}]')
print(f'GT range: [{gt_sampled.min().item():.6f}, {gt_sampled.max().item():.6f}]')
print(f'')
print('=== Gradient Analysis ===')
for name, param in model.named_parameters():
    if param.grad is not None:
        grad_norm = param.grad.norm().item()
        grad_mean = param.grad.abs().mean().item()
        grad_max = param.grad.abs().max().item()
        param_norm = param.data.norm().item()
        print(f'  {name:50s} | grad_norm={grad_norm:.6f} | grad_max={grad_max:.6f} | param_norm={param_norm:.4f}')
    else:
        print(f'  {name:50s} | NO GRADIENT')

In [ ]:
# 梯度可视化
grad_info = []
for name, param in model.named_parameters():
    if param.grad is not None:
        short_name = name.replace('temporal.', 'EMA.').replace('decoder.net.', 'MLP.')
        grad_info.append({
            'name': short_name,
            'grad_norm': param.grad.norm().item(),
            'grad_max': param.grad.abs().max().item(),
            'param_norm': param.data.norm().item(),
        })

if grad_info:
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 5))
    
    names = [g['name'] for g in grad_info]
    y_pos = range(len(names))
    
    grad_norms = [g['grad_norm'] for g in grad_info]
    ax1.barh(y_pos, grad_norms)
    ax1.set_yticks(y_pos)
    ax1.set_yticklabels(names, fontsize=8)
    ax1.set_xlabel('Gradient L2 Norm')
    ax1.set_title('Gradient Norms per Layer')
    ax1.set_xscale('log')
    
    param_norms = [g['param_norm'] for g in grad_info]
    ax2.barh(y_pos, param_norms)
    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(names, fontsize=8)
    ax2.set_xlabel('Parameter L2 Norm')
    ax2.set_title('Parameter Norms per Layer')
    
    plt.tight_layout()
    plt.show()

# 特别关注: density bias 的梯度
density_bias = model.decoder.net[-1].bias
print(f'Density bias value: {density_bias.data.tolist()}')
if density_bias.grad is not None:
    print(f'Density bias gradient: {density_bias.grad.tolist()}')

In [ ]:
# 对比：不带 skip connection 的前向
# 如果没有 current_action 输入，模型输出有什么变化？
with torch.no_grad():
    pts_grid = grid_tensor.unsqueeze(1)  # (N, 1, 3)
    s_exp = physics_state.expand(len(grid_tensor), -1)
    
    # 有 action skip
    a_exp = current_action.expand(len(grid_tensor), -1)
    raw_with_action = model.decode_spatial(pts_grid, s_exp, a_exp).squeeze(1)
    
    # 没有 action skip (模拟旧行为)
    raw_no_action = model.decode_spatial(pts_grid, s_exp, None).squeeze(1)

density_with = raw_with_action[:, 1].cpu().numpy()
density_without = raw_no_action[:, 1].cpu().numpy()

print('=== Skip Connection Impact ===')
print(f'With action skip:    density [{density_with.min():.4f}, {density_with.max():.4f}], '
      f'positive: {(density_with > 0).sum()}/{len(density_with)}')
print(f'Without action skip: density [{density_without.min():.4f}, {density_without.max():.4f}], '
      f'positive: {(density_without > 0).sum()}/{len(density_without)}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].hist(density_with, bins=50, alpha=0.7, label='With action skip')
axes[0].hist(density_without, bins=50, alpha=0.7, label='Without action skip')
axes[0].axvline(x=0, color='r', linestyle='--')
axes[0].set_title('Density Distribution')
axes[0].legend()
axes[0].set_yscale('log')

vis_with = raw_with_action[:, 0].cpu().numpy()
vis_without = raw_no_action[:, 0].cpu().numpy()
axes[1].hist(vis_with, bins=50, alpha=0.7, label='With action skip')
axes[1].hist(vis_without, bins=50, alpha=0.7, label='Without action skip')
axes[1].set_title('Visibility Distribution')
axes[1].legend()
axes[1].set_yscale('log')

plt.tight_layout()
plt.show()

## 7. 诊断总结

根据以上分析，检查以下关键指标：

| 检查项 | 正常范围 | 当前状态 |
|--------|---------|--------|
| EMA decays | 0.2~0.95 分散 | 全为 1.0？ |
| Density bias | 训练后应 > 0 | ？ |
| Density > 0 的点 | 前景区域多 | ？ |
| 前景射线梯度 | 非零 | ？ |
| Near/Far 覆盖 | 包含机器人 | ？ |